# Formative 3: Deep Q-Learning (DQN) Baseline - Private Eye Atari

## Group Members:
- **Yvette Gahamanyi** - Parameter Variation: [Higher]
- **Mariam Awini Issah** - Parameter Variation: [Lower]
- **Dennis Mwai Kimiri** - Parameter Variation: [Average]

## Objective:
Train a DQN agent to play the Private Eye Atari environment and conduct 10 hyperparameter tuning experiments per group member to identify optimal configurations.

### Key Hyperparameters to Tune:
- **Learning Rate (lr)**: Controls the speed of learning
- **Gamma (γ)**: Discount factor for future rewards
- **Batch Size**: Number of experiences sampled per update
- **Epsilon**: Exploration parameters (start, end, decay)

## 1. Environment Setup and Dependencies

In [15]:
# Install required packages
import subprocess
import sys

print("Installing required packages...")
packages = ["stable-baselines3", "gymnasium", "ale-py", "matplotlib", "pandas", "numpy"]

for package in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} installed")
    except:
        print(f"⚠ {package} installation attempted")

print("\n✓ All packages installed!\n")

# Now import
import ale_py
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
import warnings
warnings.filterwarnings('ignore')

# Register Atari environments
gym.register_envs(ale_py)

print("Dependencies imported successfully!")

Installing required packages...
✓ stable-baselines3 installed
✓ gymnasium installed
✓ ale-py installed
✓ matplotlib installed
✓ pandas installed
✓ numpy installed

✓ All packages installed!

Dependencies imported successfully!


In [16]:
# Verify environment is available
print("Available Atari environments:")
env_id = "ALE/PrivateEye-v5"
print(f"Using: {env_id}")

# Create and test the environment
test_env = make_atari_env(env_id, n_envs=1, seed=0)
test_env = VecFrameStack(test_env, n_stack=4)
print(f"Environment created successfully!")
print(f"Action space: {test_env.action_space}")
print(f"Observation space: {test_env.observation_space}")
test_env.close()

Available Atari environments:
Using: ALE/PrivateEye-v5
Environment created successfully!
Action space: Discrete(18)
Observation space: Box(0, 255, (84, 84, 4), uint8)


## 2. Baseline DQN Agent Configuration

In [17]:
# Baseline Hyperparameters shared across all group members
BASELINE_CONFIG = {
    "policy": "CnnPolicy",  # Options: "CnnPolicy" or "MlpPolicy"
    "learning_rate": 1e-4,
    "buffer_size": 50000,
    "learning_starts": 10000,
    "batch_size": 32,
    "tau": 1.0,
    "gamma": 0.99,  # Discount factor
    "train_freq": 4,
    "gradient_steps": 1,
    "exploration_fraction": 0.1,
    "exploration_final_eps": 0.01,
    "target_update_interval": 1000,
    "verbose": 1,
}

print("Baseline Configuration:")
for key, value in BASELINE_CONFIG.items():
    print(f"  {key}: {value}")

print("\n Policy Comparison:")
print("  - CnnPolicy: Better for image-based Atari environments (processes visual state)")
print("  - MlpPolicy: Better for low-dimensional state spaces")
print("  → Recommendation for PrivateEye: CnnPolicy (visual agent needs vision-based learning)")

Baseline Configuration:
  policy: CnnPolicy
  learning_rate: 0.0001
  buffer_size: 50000
  learning_starts: 10000
  batch_size: 32
  tau: 1.0
  gamma: 0.99
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  target_update_interval: 1000
  verbose: 1

 Policy Comparison:
  - CnnPolicy: Better for image-based Atari environments (processes visual state)
  - MlpPolicy: Better for low-dimensional state spaces
  → Recommendation for PrivateEye: CnnPolicy (visual agent needs vision-based learning)


## 3. Training Function (Shared Baseline)

In [18]:
def train_dqn_agent(config, model_name="dqn_model", total_timesteps=100000):
    """
    Train a DQN agent on PrivateEye environment.
    
    Args:
        config (dict): Hyperparameter configuration
        model_name (str): Name to save the model
        total_timesteps (int): Total training timesteps
    
    Returns:
        model: Trained DQN agent
    """
    print(f"\n{'='*60}")
    print(f"Training DQN Agent with Configuration:")
    print(f"{'='*60}")
    for key, value in config.items():
        print(f"  {key:.<30} {value}")
    
    # Create environment
    env = make_atari_env("ALE/PrivateEye-v5", n_envs=1, seed=0)
    env = VecFrameStack(env, n_stack=4)
    
    # Create DQN model
    model = DQN(
        policy=config["policy"],
        env=env,
        learning_rate=config["learning_rate"],
        buffer_size=config["buffer_size"],
        learning_starts=config["learning_starts"],
        batch_size=config["batch_size"],
        tau=config["tau"],
        gamma=config["gamma"],
        train_freq=config["train_freq"],
        gradient_steps=config["gradient_steps"],
        exploration_fraction=config["exploration_fraction"],
        exploration_final_eps=config["exploration_final_eps"],
        target_update_interval=config["target_update_interval"],
        verbose=1,
        tensorboard_log="./dqn_logs/"
    )
    
    # Train the agent
    print(f"\nTraining for {total_timesteps} timesteps...")
    model.learn(total_timesteps=total_timesteps, tb_log_name=model_name)
    
    # Save the model
    model.save(model_name)
    print(f"\n✓ Model saved as {model_name}.zip")
    
    env.close()
    return model

# Train baseline model (optional - uncomment to run)
# baseline_model = train_dqn_agent(BASELINE_CONFIG, "dqn_baseline", total_timesteps=50000)

In [19]:
## Run Baseline Training (Optional)

# Uncomment the line below to train the baseline model
# This will train for 50000 timesteps (you can increase for better results)

print("Ready to train baseline model!")
print("Uncomment the line below to start training:\n")
print("# baseline_model = train_dqn_agent(BASELINE_CONFIG, 'dqn_baseline', total_timesteps=50000)")
print("\n✓ Baseline ready. Individual members should use their experiment notebooks instead.")

Ready to train baseline model!
Uncomment the line below to start training:

# baseline_model = train_dqn_agent(BASELINE_CONFIG, 'dqn_baseline', total_timesteps=50000)

✓ Baseline ready. Individual members should use their experiment notebooks instead.


## 4. Hyperparameter Tuning Framework

**Each group member will test 10 different hyperparameter combinations.**

Variations to explore:
- **Member 1 (Yvette Gahamanyi)**: [Higher] parameters
- **Member 2 (Mariam Awini Issah)**: [Lower] parameters  
- **Member 3 (Dennis Mwai Kimiri)**: [Average] parameters

In [20]:
# Define parameter ranges for hyperparameter tuning
PARAMETER_RANGES = {
    "learning_rate": [1e-5, 5e-5, 1e-4, 5e-4, 1e-3],          # From very low to high
    "gamma": [0.90, 0.95, 0.99, 0.995],                       # Discount factor
    "batch_size": [16, 32, 64, 128],                          # Batch sizes
    "exploration_final_eps": [0.001, 0.01, 0.05, 0.1],        # Min exploration
    "target_update_interval": [500, 1000, 2000, 5000],        # Update frequency
    "buffer_size": [30000, 50000, 100000],                    # Memory buffer
}

print("Parameter Ranges for Tuning:")
print("="*60)
for param, values in PARAMETER_RANGES.items():
    print(f"{param:.<30} {values}")

# Create experiment template
EXPERIMENT_TEMPLATE = {
    "config": {},
    "model_name": "",
    "total_timesteps": 50000,  # Use lower timesteps for faster experiments
    "notes": ""
}

print("\nExperiment structure created. Each member will fill in 10 experiments.")

Parameter Ranges for Tuning:
learning_rate................. [1e-05, 5e-05, 0.0001, 0.0005, 0.001]
gamma......................... [0.9, 0.95, 0.99, 0.995]
batch_size.................... [16, 32, 64, 128]
exploration_final_eps......... [0.001, 0.01, 0.05, 0.1]
target_update_interval........ [500, 1000, 2000, 5000]
buffer_size................... [30000, 50000, 100000]

Experiment structure created. Each member will fill in 10 experiments.


## 5. Individual Member Experiments

### Each member should fill in their 10 experiments below

**MEMBER NAME: ________________________**

Fill in your 10 hyperparameter configurations and observed behaviors:

In [21]:
# Quick Reference: Example Parameter Sets for Testing

print("TEST CONFIGURATION")
print("="*70)

# Test Configuration
test_config = {
    "learning_rate": 1.5e-4,
    "gamma": 0.97,
    "batch_size": 48,
    "exploration_final_eps": 0.005,
    "buffer_size": 75000
}

print("\nTest Configuration:")
for k, v in test_config.items():
    print(f"   {k:.<30} {v}")

print("\n" + "="*70)
print("To run a test experiment, use:")
print("  result = run_experiment(test_config, 'Test Name', 1, total_timesteps=50000)")

TEST CONFIGURATION

Test Configuration:
   learning_rate................. 0.00015
   gamma......................... 0.97
   batch_size.................... 48
   exploration_final_eps......... 0.005
   buffer_size................... 75000

To run a test experiment, use:
  result = run_experiment(test_config, 'Test Name', 1, total_timesteps=50000)


In [22]:
# Create a tracking DataFrame for experiments
experiments_data = {
    "Exp_Num": list(range(1, 11)),
    "Learning_Rate": [None]*10,
    "Gamma": [None]*10,
    "Batch_Size": [None]*10,
    "Epsilon_Final": [None]*10,
    "Buffer_Size": [None]*10,
    "Avg_Reward": [None]*10,
    "Max_Reward": [None]*10,
    "Observed_Behavior": [""]*10
}

experiments_df = pd.DataFrame(experiments_data)

print("Experiment Tracking Template:")
print(experiments_df.to_string(index=False))
print("\n✓ Use this template to track your 10 experiments per member")

# Function to update experiment results
def update_experiment_result(df, exp_num, lr, gamma, batch_size, epsilon_final, buffer_size, avg_reward, max_reward, behavior):
    """
    Update a single experiment result in the tracking dataframe.
    
    Args:
        df: Experiments dataframe
        exp_num: Experiment number (1-10)
        lr: Learning rate
        gamma: Gamma (discount factor)
        batch_size: Batch size
        epsilon_final: Epsilon final (exploration)
        buffer_size: Buffer size
        avg_reward: Average reward
        max_reward: Max reward
        behavior: Observed behavior notes
    """
    idx = exp_num - 1
    df.loc[idx, "Learning_Rate"] = lr
    df.loc[idx, "Gamma"] = gamma
    df.loc[idx, "Batch_Size"] = batch_size
    df.loc[idx, "Epsilon_Final"] = epsilon_final
    df.loc[idx, "Buffer_Size"] = buffer_size
    df.loc[idx, "Avg_Reward"] = avg_reward
    df.loc[idx, "Max_Reward"] = max_reward
    df.loc[idx, "Observed_Behavior"] = behavior
    print(f"✓ Experiment {exp_num} updated")
    return df

# Example usage:
# experiments_df = update_experiment_result(
#     experiments_df, 
#     exp_num=1,
#     lr=5e-5,
#     gamma=0.95,
#     batch_size=16,
#     epsilon_final=0.001,
#     buffer_size=30000,
#     avg_reward=150.5,
#     max_reward=250.0,
#     behavior="Stable convergence, slow early learning"
# )

Experiment Tracking Template:
 Exp_Num Learning_Rate Gamma Batch_Size Epsilon_Final Buffer_Size Avg_Reward Max_Reward Observed_Behavior
       1          None  None       None          None        None       None       None                  
       2          None  None       None          None        None       None       None                  
       3          None  None       None          None        None       None       None                  
       4          None  None       None          None        None       None       None                  
       5          None  None       None          None        None       None       None                  
       6          None  None       None          None        None       None       None                  
       7          None  None       None          None        None       None       None                  
       8          None  None       None          None        None       None       None                  
       9        

In [23]:
def get_experiment_summary(df):
    """
    Generate a summary of experiment results.
    
    Args:
        df: Experiments dataframe
    
    Returns:
        Dictionary with summary statistics
    """
    valid_rewards = df["Avg_Reward"].dropna()
    
    if len(valid_rewards) == 0:
        print("No experiment results yet")
        return None
    
    summary = {
        "total_experiments_run": len(valid_rewards),
        "avg_reward_mean": valid_rewards.mean(),
        "avg_reward_std": valid_rewards.std(),
        "best_experiment": df.loc[valid_rewards.idxmax(), "Exp_Num"],
        "best_avg_reward": valid_rewards.max(),
        "worst_experiment": df.loc[valid_rewards.idxmin(), "Exp_Num"],
        "worst_avg_reward": valid_rewards.min(),
    }
    
    print("\n" + "="*60)
    print("EXPERIMENT SUMMARY")
    print("="*60)
    print(f"Total Experiments Run: {summary['total_experiments_run']}/10")
    print(f"Average Reward (mean): {summary['avg_reward_mean']:.2f} ± {summary['avg_reward_std']:.2f}")
    print(f"Best Experiment: #{summary['best_experiment']} (Avg: {summary['best_avg_reward']:.2f})")
    print(f"Worst Experiment: #{summary['worst_experiment']} (Avg: {summary['worst_avg_reward']:.2f})")
    print("="*60 + "\n")
    
    return summary

def export_results_to_csv(df, member_name):
    """
    Export experiment results to CSV file.
    
    Args:
        df: Experiments dataframe
        member_name: Name of the group member
    """
    filename = f"experiments_{member_name.replace(' ', '_')}.csv"
    df.to_csv(filename, index=False)
    print(f"✓ Results saved to {filename}")
    return filename

# Example usage:
# summary = get_experiment_summary(experiments_df)
# export_results_to_csv(experiments_df, "Mariam Awini Issah")

### Example: How to structure your experiments

For each experiment, create a configuration by modifying the baseline, then train and evaluate:
- **Experiment 1**: Lower learning rate (test convergence with slower learning)
- **Experiment 2**: Higher learning rate (test faster learning vs stability)
- **Experiment 3**: Lower gamma (test short-term vs long-term reward balance)
- **Experiment 4**: Higher gamma (test focus on long-term rewards)
- **Experiment 5**: Smaller batch size (test sample variance impact)
- **Experiment 6**: Larger batch size (test training stability)
- **Experiment 7**: Lower exploration final eps (more exploitation)
- **Experiment 8**: Higher exploration final eps (more exploration)
- **Experiment 9**: Different buffer size (test memory capacity)
- **Experiment 10**: Combined optimal changes from experiments 1-9

## Quick Test: Run a Single Experiment

```python
test_result = run_experiment(test_config, "Test Name", 1, total_timesteps=10000)
```

*Note: Use smaller `total_timesteps` (e.g., 10000-30000) for quick testing before running full experiments (50000+)*

In [24]:
# QUICK TEST (Uncomment to run a short test)
# This will train for only 10000 timesteps (~2-3 minutes) to verify everything works

print("To run a quick test, uncomment the line below:")
print("\n# test_result = run_experiment(test_config, 'Test Name', 1, total_timesteps=10000)")

To run a quick test, uncomment the line below:

# test_result = run_experiment(test_config, 'Test Name', 1, total_timesteps=10000)


In [25]:
def run_experiment(experiment_config, member_name, exp_num, total_timesteps=50000):
    """
    Run a single hyperparameter experiment and evaluate the trained model.
    
    Args:
        experiment_config (dict): Modified configuration for this experiment
        member_name (str): Name of the group member
        exp_num (int): Experiment number
        total_timesteps (int): Training timesteps
    
    Returns:
        dict: Results including average reward, max reward, and observations
    """
    print(f"\n{'='*70}")
    print(f"EXPERIMENT {exp_num} - {member_name}")
    print(f"{'='*70}")
    print("Configuration:")
    for key, value in experiment_config.items():
        print(f"  {key:.<30} {value}")
    
    # Merge with baseline config
    config = {**BASELINE_CONFIG, **experiment_config}
    model_name = f"dqn_{member_name.replace(' ', '_')}_exp{exp_num}"
    
    try:
        # Train model
        model = train_dqn_agent(config, model_name, total_timesteps)
        
        # Evaluate the model
        env = make_atari_env("ALE/PrivateEye-v5", n_envs=1, seed=42)
        env = VecFrameStack(env, n_stack=4)
        
        rewards = []
        num_eval_episodes = 3
        
        for _ in range(num_eval_episodes):
            obs, _ = env.reset()
            episode_reward = 0
            done = False
            while not done:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                episode_reward += reward
                done = terminated or truncated
            rewards.append(episode_reward)
        
        env.close()
        
        avg_reward = np.mean(rewards)
        max_reward = np.max(rewards)
        
        print(f"\nEvaluation Results:")
        print(f"  Average Reward (3 episodes): {avg_reward:.2f}")
        print(f"  Max Reward: {max_reward:.2f}")
        
        return {
            "avg_reward": avg_reward,
            "max_reward": max_reward,
            "model_name": model_name,
            "status": "✓ Success"
        }
    
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        return {
            "avg_reward": 0,
            "max_reward": 0,
            "model_name": model_name,
            "status": f"✗ Failed: {str(e)}"
        }

# Example usage (uncomment to run a specific experiment):
# experiment_1_config = {
#     "learning_rate": 5e-5,  # Lower LR
#     "gamma": 0.99,
#     "batch_size": 32,
# }
# results = run_experiment(experiment_1_config, "Mariam Awini Issah", 1)

## 6. Results Aggregation and Analysis

In [26]:
# Template for consolidated results table
# After all experiments are done, fill this with actual results

CONSOLIDATED_RESULTS = pd.DataFrame({
    "Member": ["Yvette Gahamanyi"]*10 + ["Mariam Awini Issah"]*10 + ["Dennis Mwai Kimiri"]*10,
    "Exp_Num": list(range(1,11)) * 3,
    "Learning_Rate": [None]*30,
    "Gamma": [None]*30,
    "Batch_Size": [None]*30,
    "Epsilon_Final": [None]*30,
    "Avg_Reward": [None]*30,
    "Max_Reward": [None]*30,
    "Observed_Behavior": [None]*30
})

print("Consolidated Results Table Template (30 total experiments):")
print(f"\nRows: {len(CONSOLIDATED_RESULTS)}")
print(f"Columns: {list(CONSOLIDATED_RESULTS.columns)}")
print("\nAfter completing all experiments, this table will show:")
print("  - Which hyperparameter changes improved performance")
print("  - Which changes harmed performance")
print("  - Final optimal configuration per member")

Consolidated Results Table Template (30 total experiments):

Rows: 30
Columns: ['Member', 'Exp_Num', 'Learning_Rate', 'Gamma', 'Batch_Size', 'Epsilon_Final', 'Avg_Reward', 'Max_Reward', 'Observed_Behavior']

After completing all experiments, this table will show:
  - Which hyperparameter changes improved performance
  - Which changes harmed performance
  - Final optimal configuration per member


In [27]:
def visualize_results(results_df):
    """
    Visualize hyperparameter tuning results.
    
    Args:
        results_df (pd.DataFrame): Results dataframe with hyperparameters and performance
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Average Reward by Member
    if "Member" in results_df.columns and "Avg_Reward" in results_df.columns:
        member_rewards = results_df.groupby("Member")["Avg_Reward"].mean()
        member_rewards.plot(kind="bar", ax=axes[0, 0], color="skyblue")
        axes[0, 0].set_title("Average Reward by Member")
        axes[0, 0].set_ylabel("Avg Reward")
        axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Plot 2: Learning Rate Impact
    if "Learning_Rate" in results_df.columns and "Avg_Reward" in results_df.columns:
        results_df.plot(x="Learning_Rate", y="Avg_Reward", kind="scatter", ax=axes[0, 1], color="green")
        axes[0, 1].set_title("Learning Rate Impact on Reward")
        axes[0, 1].set_xlabel("Learning Rate")
        axes[0, 1].set_ylabel("Avg Reward")
    
    # Plot 3: Gamma Impact
    if "Gamma" in results_df.columns and "Avg_Reward" in results_df.columns:
        results_df.plot(x="Gamma", y="Avg_Reward", kind="scatter", ax=axes[1, 0], color="orange")
        axes[1, 0].set_title("Gamma Impact on Reward")
        axes[1, 0].set_xlabel("Gamma (Discount Factor)")
        axes[1, 0].set_ylabel("Avg Reward")
    
    # Plot 4: Batch Size Impact
    if "Batch_Size" in results_df.columns and "Avg_Reward" in results_df.columns:
        results_df.plot(x="Batch_Size", y="Avg_Reward", kind="scatter", ax=axes[1, 1], color="red")
        axes[1, 1].set_title("Batch Size Impact on Reward")
        axes[1, 1].set_xlabel("Batch Size")
        axes[1, 1].set_ylabel("Avg Reward")
    
    plt.tight_layout()
    plt.show()

# Example usage (uncomment after filling in results):
# visualize_results(CONSOLIDATED_RESULTS)

## 7. Agent Evaluation (play.py)

The `play.py` script loads the best trained model and evaluates it with:
- **GreedyQPolicy**: Selects actions with the highest Q-value (no exploration)
- **Real-time Rendering**: Shows the agent playing the PrivateEye game
- **Performance Statistics**: Tracks rewards and episode lengths

### To use play.py:
```bash
python play.py
```

This will:
1. Load `dqn_model.zip`
2. Run 5 evaluation episodes
3. Display game visualization
4. Print episode statistics

## 8. Key Insights for Presentation

Each member should prepare answers to these questions for the 2-minute presentation:

### For Individual Member Presentations (2 minutes max each):

1. **Summary of 10 Experiments**
   - Brief description of what parameters were varied and why
   
2. **Which Parameter Changes Improved Performance?**
   - Analyze which hyperparameter modifications led to better rewards
   - Explain the mechanism behind the improvement
   
3. **Which Changes Harmed Performance?**
   - Identify hyperparameter combinations that decreased performance
   - Discuss potential reasons
   
4. **Final Optimal Configuration**
   - Which experiment achieved the best results?
   - Why does this configuration work best for PrivateEye?
   - How different is it from the baseline?

### General Group Questions (Prepare all members):

- **Trade-offs in Hyperparameter Choices**: What is the balance between exploration and exploitation in your best model?
- **Model Behavior**: Why does your final model behave the way it does? How does it navigate PrivateEye?
- **Policy Architecture Decision**: Why did you choose CnnPolicy over MlpPolicy? (Hint: Visual environment!)

### Supporting Evidence:
- Present your results table
- Show hyperparameter comparison charts
- Play a short gameplay clip from `play.py` output

## 9. References and Additional Notes

### DQN Hyperparameter Explanations:

| Hyperparameter | Role | Effect if Increased | Effect if Decreased |
|---|---|---|---|
| **Learning Rate** | Speed of weight updates | Faster learning, risk of instability | Slower convergence, more stable |
| **Gamma** | Future reward discount | Values long-term rewards more | Focuses on immediate rewards |
| **Batch Size** | Samples per update | More stable but slower | Faster updates, more variance |
| **Epsilon Final** | Min exploration rate | More exploration always | More exploitation after training |
| **Buffer Size** | Experience memory | More diverse experiences | Lower memory, more correlated data |
| **Target Update** | Q-network sync frequency | Faster adaptation | More stable but slower learning |

### PrivateEye Environment Context:
- **Visual-based navigation**: Agent must understand and navigate complex maze environments
- **Sparse rewards**: Rewards are infrequent, requiring patience and long-term planning
- **High-dimensional input**: Raw pixel observations → CNN essential for feature extraction
- **Challenge**: Requires both exploration (find goals) and exploitation (navigate efficiently)

### Submission Checklist:
- [ ] All 3 members completed 10 experiments each (30 total)
- [ ] Results documented in consolidated table
- [ ] `train.py` trains and saves models successfully
- [ ] `play.py` loads models and renders gameplay
- [ ] README updated with team info and results table
- [ ] Video of agent playing included in submission
- [ ] GitHub repository contains all files
- [ ] Group presentation slot booked with coach (Week 6)